<a href="https://colab.research.google.com/github/oselumeseagbonrofo/small-llm-experiments/blob/main/full_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Steps to fully finetune a pretained LLM

## Load dataset

This example uses a Question Answering one. For datasets package documentation see: https://github.com/huggingface/datasets

For viewing datasets on Hugging Face: https://huggingface.co/datasets/

QA dataset here should have 3 fields: question, context, answers

In [1]:
from datasets import load_dataset
training_set = load_dataset("vincentkoc/tiny_qa_benchmark_pp", split="train")

README.md:   0%|          | 0.00/6.91k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

core_en.jsonl:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

pack_de_40.jsonl:   0%|          | 0.00/14.5k [00:00<?, ?B/s]

pack_en_10.jsonl:   0%|          | 0.00/2.77k [00:00<?, ?B/s]

...

pack_zh-CN_40.jsonl:   0%|          | 0.00/14.5k [00:00<?, ?B/s]

pack_es_40.jsonl:   0%|          | 0.00/15.4k [00:00<?, ?B/s]

sup-ancientlang_en_10.jsonl:   0%|          | 0.00/3.65k [00:00<?, ?B/s]

sup-medicine_en_10.jsonl:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/662 [00:00<?, ? examples/s]

## Split train and test set

In [2]:
training_set = training_set.train_test_split(test_size=0.2)

## Load DistilBERT tokenizer to process training set's question and context fields

In [3]:
from transformers import AutoTokenizer
model_name = "distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

## Preprocess data function

In [4]:
def preprocess_function(examples):
 questions = [q.strip() for q in examples["text"]]
 inputs = tokenizer(
     questions,
     examples["context"],
     max_length=384,
     truncation="only_second",
     return_offsets_mapping=True,
     padding="max_length",
     )
 offset_mapping = inputs.pop("offset_mapping")
 answers = examples["label"]
 contexts = examples["context"]
 start_positions = []
 end_positions = []
 for i, offset in enumerate(offset_mapping):
  answer = answers[i]
  context = contexts[i]
  start_char = context.find(answer)
  end_char = start_char + len(answer)
  sequence_ids = inputs.sequence_ids(i)

  idx = 0
  while sequence_ids[idx] != 1:
    idx += 1
  context_start = idx
  while sequence_ids[idx] == 1:
    idx += 1
  context_end = idx - 1

  # If the answer is not fully inside the context, label it (0, 0)
  if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
    start_positions.append(0)
    end_positions.append(0)
  else:
    idx = context_start
    while idx <= context_end and offset[idx][0] <= start_char:
      idx += 1
    start_positions.append(idx - 1)

    idx = context_end
    while idx >= context_start and offset[idx][1] >= end_char:
      idx -= 1
    end_positions.append(idx + 1)
 inputs["start_positions"] = start_positions
 inputs["end_positions"] = end_positions
 return inputs

In [5]:
tokenized_dataset = training_set.map(preprocess_function, batched=True,
                                     remove_columns=training_set["train"].column_names)

Map:   0%|          | 0/529 [00:00<?, ? examples/s]

Map:   0%|          | 0/133 [00:00<?, ? examples/s]

## Load Model

In [6]:
from transformers import AutoModelForQuestionAnswering

model = AutoModelForQuestionAnswering.from_pretrained(model_name)

model.safetensors: reconstructing file:   0%|          |  0.00B /  542MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
from transformers import DefaultDataCollator
data_collator = DefaultDataCollator()

## Create training arguments

In [12]:
from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
 output_dir="my_awesome_qa_model",
 eval_strategy="epoch",
 learning_rate=2e-5,
 per_device_train_batch_size=16,
 per_device_eval_batch_size=16,
 num_train_epochs=5,
 weight_decay=0.01,
 push_to_hub=False,
)

## Create Trainer object

In [13]:
trainer = Trainer(
 model=model,
 args=training_args,
 train_dataset=tokenized_dataset["train"],
 eval_dataset=tokenized_dataset["test"],
 data_collator=data_collator,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.546119
2,No log,0.579445
3,No log,0.560355
4,No log,0.608884
5,No log,0.605803


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=170, training_loss=0.21965394861557905, metrics={'train_runtime': 132.1693, 'train_samples_per_second': 20.012, 'train_steps_per_second': 1.286, 'total_flos': 259183087188480.0, 'train_loss': 0.21965394861557905, 'epoch': 5.0})

## Example with trained model

In [15]:
question = "How many titles has Juventus won?"
context = """Juventus Football Club (from Latin: iuventūs), colloquially
 known as Juve, is a professional football club based in Turin, Piedmont,
Italy, that competes in the Serie A, the top tier of the Italian football
league system. Nicknamed la Vecchia Signora (the Old Lady), the
club has won 36 official league titles, 14 Coppa Italia titles and nine
Supercoppa Italiana titles, being the record holder for all these
competitions;"""

inputs = tokenizer(question, context, return_tensors="pt")
import torch
with torch.no_grad():
 outputs = model(**inputs.to(model.device))

answer_start_index = outputs.start_logits.argmax()
answer_end_index = outputs.end_logits.argmax()

predict_answer_tokens = inputs.input_ids[0, answer_start_index : answer_end_index + 1]
tokenizer.decode(predict_answer_tokens)

'14 Coppa Italia titles and nine Supercoppa Italiana titles'